Possible open-source datasets that are relevant to your research area.
We explored the following open source datasets for our MLEngine
* kmv
* aeoio
* knvso


Identifying variables that would serve as the expected output of our final application


We decided to go with this dataset because....


since the model we're working on is a computer vision model and the dataset we've chosen contains images and labels of those images seperately, we're first going to form a single csv file by writing a pyhon script to contain the image names, image labels as well as specific columns present in each label i.e. class_id,x_center,y_center,width,height
in this case, some images may have different potholes and each having their own labels, so we'll have say class_id(1),... class_id(2)...

In [2]:
!pip install pandas

Defaulting to user installation because normal site-packages is not writeable
  Using cached pandas-3.0.5-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (79 kB)
  Using cached numpy-2.5.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl.metadata (6.6 kB)
Using cached pandas-3.0.5-cp314-cp314-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (11.0 MB)
Using cached numpy-2.5.3-cp314-cp314-manylinux_2_27_x86_64.manylinux_2_28_x86_64.whl (16.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [pandas]2m1/2 [pandas]


In [3]:
"""
This script combines the images and their corresponding label files
from the train, valid, and test folders into a single CSV file.

For each image, the CSV stores the image name and the dataset split.
The label files contain annotations in the format:
class_id, x_center, y_center, width, height.

Since an image can contain multiple potholes, each annotation is
stored in its own set of columns, such as:
class_id(1), x_center(1), y_center(1), width(1), height(1),
class_id(2), x_center(2), y_center(2), width(2), height(2), and so on.

The script first finds the highest number of annotations in any
label file and creates enough columns to accommodate all of them.
If an image has fewer annotations, the remaining columns are left empty.

The completed CSV file is saved in the data/ directory.
"""



import os
import pandas as pd

# Location of the dataset
data_dir = "data/1"

# The dataset is divided into these three folders
splits = ["train", "valid", "test"]

# This is where the generated CSV will be saved
output_file = "data/pothole_annotations.csv"


# First, go through all the label files and find the image
# that contains the highest number of pothole annotations.
max_annotations = 0
total_images = 0
total_annotations = 0

for split in splits:
    images_dir = os.path.join(data_dir, split, "images")
    labels_dir = os.path.join(data_dir, split, "labels")

    image_files = [
        file for file in os.listdir(images_dir)
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    ]

    total_images += len(image_files)

    for image_file in image_files:
        image_name = os.path.splitext(image_file)[0]
        label_file = os.path.join(labels_dir, image_name + ".txt")

        if not os.path.exists(label_file):
            continue

        with open(label_file, "r") as file:
            annotations = [
                line.strip()
                for line in file
                if line.strip()
            ]

        number_of_annotations = len(annotations)
        total_annotations += number_of_annotations

        if number_of_annotations > max_annotations:
            max_annotations = number_of_annotations


print(f"Total images: {total_images}")
print(f"Total annotations: {total_annotations}")
print(f"Maximum potholes in one image: {max_annotations}")


# Create the CSV columns based on the maximum number of
# potholes found in a single image.
columns = ["image_name", "split"]

for i in range(1, max_annotations + 1):
    columns.extend([
        f"class_id({i})",
        f"x_center({i})",
        f"y_center({i})",
        f"width({i})",
        f"height({i})"
    ])


# Read each image and its corresponding label file.
# All annotations belonging to an image will be stored
# in the same row.
rows = []

for split in splits:
    images_dir = os.path.join(data_dir, split, "images")
    labels_dir = os.path.join(data_dir, split, "labels")

    image_files = [
        file for file in os.listdir(images_dir)
        if file.lower().endswith((".jpg", ".jpeg", ".png", ".bmp", ".webp"))
    ]

    print(f"Processing {split}: {len(image_files)} images")

    for image_file in image_files:
        image_name = os.path.splitext(image_file)[0]
        label_file = os.path.join(labels_dir, image_name + ".txt")

        # Store the image name and the dataset split
        row = {
            "image_name": image_file,
            "split": split
        }

        if os.path.exists(label_file):
            with open(label_file, "r") as file:
                annotations = [
                    line.strip()
                    for line in file
                    if line.strip()
                ]

            # Each line in the label file represents one pothole
            for i, annotation in enumerate(annotations, start=1):
                values = annotation.split()

                # YOLO annotation format:
                # class_id, x_center, y_center, width, height
                if len(values) != 5:
                    print(f"Invalid annotation in: {label_file}")
                    continue

                class_id, x_center, y_center, width, height = values

                row[f"class_id({i})"] = int(class_id)
                row[f"x_center({i})"] = float(x_center)
                row[f"y_center({i})"] = float(y_center)
                row[f"width({i})"] = float(width)
                row[f"height({i})"] = float(height)

        rows.append(row)


# Convert the collected rows into a DataFrame.
# Images with fewer potholes will have empty values
# in the columns for the missing annotations.
df = pd.DataFrame(rows, columns=columns)


# Save the completed dataset as a CSV file.
df.to_csv(output_file, index=False)


print(f"\nCSV saved to: {output_file}")
print(f"Rows: {len(df)}")
print(f"Columns: {len(df.columns)}")

print("\nImages by split:")
print(df["split"].value_counts())

display(df.head())

Total images: 3940
Total annotations: 10111
Maximum potholes in one image: 21
Processing train: 3345 images
Processing valid: 397 images
Processing test: 198 images

CSV saved to: data/pothole_annotations.csv
Rows: 3940
Columns: 107

Images by split:
split
train    3345
valid     397
test      198
Name: count, dtype: int64


,image_name,split,class_id(1),x_center(1),y_center(1),width(1),height(1),class_id(2),x_center(2),y_center(2),...,class_id(20),x_center(20),y_center(20),width(20),height(20),class_id(21),x_center(21),y_center(21),width(21),height(21)
0,01_jpg.rf.311d897b199b9edde57b99bc4b606074.jpg,train,0,0.492969,0.306250,0.451562,0.223438,0.0,0.549219,0.623437,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,01_jpg.rf.bf7e43382a861a8c57127bca8c177a2d.jpg,train,0,0.492969,0.306250,0.451562,0.223438,0.0,0.549219,0.623437,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,02_jpg.rf.2788bebeb55f808442e3a7e04f8fcea8.jpg,train,0,0.482031,0.551562,0.964063,0.670312,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,02_jpg.rf.bf08db3e12adb8f7cff300bac0297ab0.jpg,train,0,0.517969,0.551562,0.964063,0.670312,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,03_jpg.rf.33e1ef5733f131973cfa5aeef0d0d625.jpg,train,0,0.514062,0.628906,0.829688,0.435156,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
# Find the image with the highest number of pothole annotations

annotation_columns = [
    column for column in df.columns
    if column.startswith("class_id(")
]

df["number_of_potholes"] = df[annotation_columns].notna().sum(axis=1)

df[df["number_of_potholes"] == 21][
    ["image_name", "split", "number_of_potholes"]
]

,image_name,split,number_of_potholes
155,17_jpg.rf.43df49bad1838757beb55a58f1eb7152.jpg,train,21
156,17_jpg.rf.c93982e46e482554cf9739ece4bb0e63.jpg,train,21
164,19_jpg.rf.c4a8c4ba9d53fbbd6533420acfc1c6bc.jpg,train,21
165,19_jpg.rf.e7a3a31239f317f95f057e510b6b5bb8.jpg,train,21
